In [1]:
from flexcraft.pipelines.tcr.utils import download_structure
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

[10:56:12] Initializing Normalizer


In [2]:
data_dir = Path("../../data/adapt/").resolve()
table_path = data_dir/"input_data/tcr3d_data/mhc1.csv"
out_dir = data_dir/f"clustering_{datetime.now().strftime('%Y-%d-%b_%H:%M:%S')}"
out_dir.mkdir()

In [3]:
table = pd.read_csv(table_path)

In [4]:
table = table[table["Bound to TCR"].astype(bool)]
table = table[table["Species"]=="Human"]
table = table[table["Resolution"].astype(float)<3]
table = table.sort_values("Release date", ascending=False)[:400]

In [5]:
table.head()

,PDB ID,MHC allele,Species,Peptide*,Bound to TCR,Release date,Pubmed,Resolution
1448,9PBH,HLA-B*27,Human,GRLPLLNPI,1.0,2026-03-18,NaN,2.13
1447,9PBG,HLA-B*27,Human,LRVMMLAPF,1.0,2026-03-18,NaN,2.00
1438,9NMY,HLA-A*02,Human,TLMSAMTNL,1.0,2026-03-18,NaN,2.01
1437,9NMX,HLA-A*02,Human,TLMSAMTNL,1.0,2026-03-18,NaN,2.19
1436,9NMW,HLA-A*02,Human,TLMSAMTNL,1.0,2026-03-18,NaN,2.11


In [6]:
def count_chains(pdb_path:Path,)->int:
    n=0
    with open(pdb_path, "r") as rf:
        l=rf.readline()
        while l:
            if l.startswith("TER"):
                n+=1
            l = rf.readline()
    return n

In [8]:
def build_database(df:pd.DataFrame, out_dir:Path, ab:bool=False, chain_number:int|None=None):
    out_dir.mkdir(exist_ok=True)
    drop=[]
    for pdb_id in df["PDB ID"].to_list():
        path = download_structure(pdb_id=pdb_id, file_format="antibody" if ab else "biological assembly", out_dir=out_dir)
        if not path is None:
            if (path.parent/(path.name+".gz")).exists():
                (path.parent/(path.name+".gz")).unlink()
            if not chain_number is None:
                chain_count = count_chains(path)
                if chain_count!=chain_number:
                    path.unlink()
                    print(f"Skipping {pdb_id} with {chain_count}!")
                    drop.append(pdb_id)
        else:
            drop.append(pdb_id)
    df=df[(df["PDB ID"].to_numpy()[:, None]==np.array(drop)[None,:]).any(axis=1)]
    df.to_csv(out_dir/"annotation.csv")
build_database(table, out_dir/"pdb_files", ab=False, chain_number=5)

Skipping 9RCV with 9!
Skipping 9K2U with 3!
Skipping 9K2T with 3!
Skipping 9K2S with 4!
Skipping 9DY8 with 3!
Skipping 9J4V with 3!
Skipping 8TMU with 4!
Encountered HTTPError for 8V4Z: HTTP Error 404: Not Found with format biological assembly
Skipping 9BL3 with 4!
Skipping 9BL2 with 4!
Encountered HTTPError for 9BL4: HTTP Error 404: Not Found with format biological assembly
Skipping 9BL5 with 4!
Skipping 9BL6 with 4!
Skipping 9BL9 with 4!
Encountered HTTPError for 8QFY: HTTP Error 404: Not Found with format biological assembly
Skipping 8EK5 with 4!
Skipping 8ES8 with 11!
Skipping 7T5M with 3!
Skipping 7SIF with 3!
Skipping 7SIH with 3!
Encountered HTTPError for 7PBC: HTTP Error 404: Not Found with format biological assembly
Encountered HTTPError for 7PDW: HTTP Error 404: Not Found with format biological assembly
Skipping 7R7Y with 3!
Skipping 7WT4 with 3!
Skipping 7WT5 with 3!
Skipping 7BH8 with 10!
Skipping 7WKJ with 3!
Skipping 7DUU with 4!
Skipping 7EJN with 3!
Skipping 7EJL with 3

In [9]:
#import os
#os.system(f"foldseek easy-cluster {out_dir/'pdb_files'} {out_dir/'cluster_result'}, $TMP -c 0.9 --gpu 1")
# execute on allocation
print(f"foldseek easy-cluster {out_dir/'pdb_files'} {out_dir/'cluster_result'} $TMP -c 0.9 --gpu 1")
from time import sleep
while not (out_dir/'cluster_result_cluster.tsv').exists():
    sleep(5)

foldseek easy-cluster /hkfs/work/workspace_haic/scratch/hgf_dsb0249-BinderDesign/flexcraft/data/adapt/clustering_2026-06-May_10:56:14/pdb_files /hkfs/work/workspace_haic/scratch/hgf_dsb0249-BinderDesign/flexcraft/data/adapt/clustering_2026-06-May_10:56:14/cluster_result $TMP -c 0.9 --gpu 1


In [10]:
clusters = pd.read_csv(out_dir/'cluster_result_cluster.tsv', delimiter="\t", header=None)
clusters.columns = ["rep", "member"]
counts = clusters["rep"].value_counts()[:4]
reps = counts.index.map(lambda x: x.split("_")[0]).to_list()
print(counts)
rep_dir = out_dir/"representative"
rep_dir.mkdir(exist_ok=True)
for pdb_id in reps:
    Path(out_dir/f"pdb_files/{pdb_id}.pdb").rename(out_dir/f"representative/{pdb_id}.pdb")

rep
2J8U_F    193
1LP9_B    184
3W0W_A    182
7N2S_D     89
Name: count, dtype: int64


In [11]:
# Develop alignment
from flexcraft.data.data import DesignData
from flexcraft.files import PDBFile
from salad.modules.utils.geometry import index_align, index_kabsch, apply_alignment

In [14]:
# get sample design 
design_path = Path("/home/hgf_dkfz/hgf_dsb0249/workspaces/haicwork/hgf_dsb0249-BinderDesign/flexcraft/data/adapt/2026-06-May_11:54:35_testing_design_0/5BS0+MAVMAPRTLV+TLMSAMTNL+CAASFGSNYK+CASSAQSTAR_0.pdb")
design = PDBFile(path=design_path).to_data()

In [ ]:
def _data_to_pos(x):
    if hasattr(x, "to_data"):
        x = x.to_data()
        assert isinstance(x, DesignData)
    if isinstance(x, DesignData):
        x = x["atom_positions"]
    return x
def order_chains(design:DesignData, order:np.ndarray):
    parts = []
    for chain in order:
        parts.append(design["chain_index"]==chain)
    return DesignData.concatenate([design[part] for part in parts], sep_chains=False)

def convert_chains(input_design:DesignData, d:dict|None=None):
    if d is None:
        d = {}
        for x,y in zip(np.sort(np.unique(input_design["chain_index"])), range(len(np.unique(input_design["chain_index"])))):
            d[int(x)]=int(y)
    print(d)
    design = input_design.update(chain_index=jnp.array([d[int(x)] for x in input_design["chain_index"]]))
    return design, d

def align(x:DesignData, y:DesignData, on_chain:str|int|list[str|int]):
    
    # convert chains to concrete integer range
    y,d = convert_chains(y)
    x,_ = convert_chains(x, d=d)

    on_chain=d[on_chain]

    # bring chains of x in same order
    y_chain_order = np.unique(y["chain_index"])
    x = order_chains(x, y_chain_order)

    # get chain mask
    on_chain = np.array(list(on_chain))
    chain_mask_x = (x["chain_index"][:,None]==on_chain[None,:]).any(axis=1)
    chain_mask_y = (y["chain_index"][:,None]==on_chain[None,:]).any(axis=1)

    # convert to 3d coordinates
    x = _data_to_pos(x)
    y = _data_to_pos(y)

    #

    return x

DesignData(data={'aa': array([ 0,  7,  8, 18, 16,  7, 10,  1,  0,  0, 12, 15, 18,  1,  8,  9,  8,
        3, 12, 18, 12,  3, 13, 15,  5, 16,  6, 15,  0, 10, 17,  3, 12,  7,
       19,  5,  4,  7,  6,  1,  9,  2,  9, 12, 16, 15, 12, 12, 17, 18,  4,
        1, 17,  7,  4,  8,  7, 16, 15,  9,  4, 12,  3,  0, 15, 12,  6,  2,
       15, 15,  9,  4, 15, 16, 12, 17, 10,  0,  0, 15,  7,  1,  9,  6, 15,
        0, 16, 17, 12, 13, 19,  0, 15, 19, 19,  5, 15,  7,  7,  1, 10,  1,
        5, 18,  5,  9,  2,  9, 16,  2, 12, 15, 18, 10,  1,  9,  0,  9, 18,
       16,  7, 16,  1,  4, 17, 12, 10,  2, 16,  4,  9,  7,  7, 18, 16, 12,
       15, 13, 15,  1, 10, 15,  9, 11,  4, 15, 18, 15, 19, 17,  7,  7, 16,
        1,  9,  7,  9, 12,  7,  5, 12,  5,  8, 17,  5, 15,  8, 16,  7,  4,
        3,  2,  9,  3,  5,  1,  9,  4,  5, 15,  9,  4,  7,  5, 15,  3, 15,
        4, 15,  8, 14,  3, 18, 15, 16, 12,  8, 12,  9,  6, 15,  0, 12, 17,
       12,  7, 19,  7, 18, 15,  7, 18,  0, 13, 16, 15,  7,  9, 15, 16,  5,
  